# Generate UMAP gif for the combined pilot plates

In [1]:
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(arrow))
suppressPackageStartupMessages(library(ggExtra))
suppressPackageStartupMessages(library(gifski))


Warning message:
“package ‘ggplot2’ was built under R version 4.2.3”
Warning message:
“package ‘arrow’ was built under R version 4.2.3”
Warning message:
“package ‘ggExtra’ was built under R version 4.2.3”
Warning message:
“package ‘gifski’ was built under R version 4.2.3”


In [2]:
pilot_data_umap_df <- read_parquet("./results/UMAP_plates_1_2_combined_KK22-05-198.parquet")

# Modify the Metadata_dose column to append 'uM' and preserve the order
pilot_data_umap_df$Metadata_dose <- paste0(pilot_data_umap_df$Metadata_dose, " uM")

# Remove " uM", sort the numeric values, and then append " uM" again
sorted_doses <- sort(as.numeric(gsub(" uM", "", pilot_data_umap_df$Metadata_dose)))

# Ensure no duplicates and restore " uM"
unique_sorted_doses <- unique(sorted_doses)
pilot_data_umap_df$Metadata_dose <- factor(
    pilot_data_umap_df$Metadata_dose,
    levels = paste0(unique_sorted_doses, " uM")
)

# Group by Metadata_Well and count cells
cell_count_df <- pilot_data_umap_df %>%
    dplyr::group_by(Metadata_Well) %>%
    dplyr::count() %>%
    dplyr::rename(Metadata_Cell_Count = n)

# Merge the cell count data with the original dataframe
pilot_data_umap_df <- pilot_data_umap_df %>%
    dplyr::left_join(cell_count_df, by = "Metadata_Well")

dim(pilot_data_umap_df)
head(pilot_data_umap_df)


[1] 40660    22

Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_ImageNumber,Metadata_Image_Count_Cells,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Nuclei_Number_Object_Number,⋯,Metadata_Well,Metadata_WellCol,Metadata_WellRow,Metadata_dose,Metadata_dose_unit,Metadata_heart_number,Metadata_treatment,UMAP0,UMAP1,Metadata_Cell_Count
<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,⋯,<chr>,<int>,<chr>,<fct>,<chr>,<int>,<chr>,<dbl>,<dbl>,<int>
238.99545,104.1795,2,2,4,1,35,220.49110,103.8781,4,⋯,A01,1,A,5 uM,uM,3,drug_x,-0.3673327,3.3385696,353
92.41599,130.3018,3,3,6,1,35,77.76578,148.8271,6,⋯,A01,1,A,5 uM,uM,3,drug_x,2.2695999,3.4131932,353
199.19100,189.8277,4,4,7,1,35,180.30225,171.3147,7,⋯,A01,1,A,5 uM,uM,3,drug_x,0.7789490,-0.2976294,353
411.24057,178.8786,5,5,8,1,35,419.32820,196.2261,8,⋯,A01,1,A,5 uM,uM,3,drug_x,2.1353924,4.3834801,353
777.62108,212.2321,6,6,9,1,35,765.91288,208.3967,9,⋯,A01,1,A,5 uM,uM,3,drug_x,2.2421830,4.3207059,353
363.69958,247.4057,7,7,10,1,35,377.83268,247.4114,10,⋯,A01,1,A,5 uM,uM,3,drug_x,2.2464592,3.8767743,353


In [ ]:
# Create a folder to save the images
dir.create("umap_pilot_plates_frames", showWarnings = FALSE)

# Get all unique doses in order
doses <- pilot_data_umap_df$Metadata_dose %>%
    unique() %>%
    sort(by = ~ as.numeric(sub(" uM", "", .)))

# Compute axis and color limits across all data
x_limits <- range(pilot_data_umap_df$UMAP0, na.rm = TRUE)
y_limits <- range(pilot_data_umap_df$UMAP1, na.rm = TRUE)
color_limits <- range(pilot_data_umap_df$Metadata_Cell_Count, na.rm = TRUE)

# Loop through doses and save each plot
for (i in seq_along(doses)) {
    p <- ggplot(
        pilot_data_umap_df %>% filter(Metadata_dose == doses[i]),
        aes(x = UMAP0, y = UMAP1, color = Metadata_Cell_Count)
    ) +
        geom_point(alpha = 0.6) +
        ggtitle(paste("Dose:", doses[i])) +
        theme_bw() +
        xlim(x_limits) +
        ylim(y_limits) +
        scale_color_continuous(low = "lightblue", high = "darkblue", limits = color_limits) +
        labs(color = "Well-level\ncell counts")

    ggsave(filename = sprintf("umap_pilot_plates_frames/frame_%02d.png", i), plot = p, width = 6, height = 5, bg = "white")
}

# Make the GIF
gifski(
    png_files = list.files("umap_pilot_plates_frames", full.names = TRUE, pattern = "*.png"),
    gif_file = "umap_pilot_plates_frames/umap_pilot_plates_by_dose.gif",
    width = 600, height = 500, delay = 0.35
)

[1] "umap_pilot_plates_frames/umap_pilot_plates_by_dose.gif"